In [51]:
# Importing necessary libraries

import pandas as pd
import geopandas as gpd
import plotly.express as px
import datetime
import imageio.v2 as imageio

In [52]:
# Loading the data

data = gpd.read_file("../01_docs/01_data/02_processed/trip_data_with_zones.gpkg")

gdf = data.to_crs(4326)

In [53]:
data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 12493 entries, 0 to 12492
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   OBJECTID         12493 non-null  int32   
 1   Shape_Leng       12493 non-null  float64 
 2   Shape_Area       12493 non-null  float64 
 3   zone             12493 non-null  str     
 4   LocationID       12493 non-null  int32   
 5   borough          12493 non-null  str     
 6   pickup_location  12491 non-null  float64 
 7   time_stamp       12491 non-null  str     
 8   avg_rides        12491 non-null  float64 
 9   geometry         12493 non-null  geometry
dtypes: float64(4), geometry(1), int32(2), str(3)
memory usage: 1.2 MB


In [54]:
data.head(5)

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,pickup_location,time_stamp,avg_rides,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,1.0,00:00:00,0.797260,"MULTIPOLYGON (((933100.918 192536.086, 933091...."
1,1,0.116357,0.000782,Newark Airport,1,EWR,1.0,00:30:00,0.536986,"MULTIPOLYGON (((933100.918 192536.086, 933091...."
2,1,0.116357,0.000782,Newark Airport,1,EWR,1.0,01:00:00,0.419178,"MULTIPOLYGON (((933100.918 192536.086, 933091...."
3,1,0.116357,0.000782,Newark Airport,1,EWR,1.0,01:30:00,0.189041,"MULTIPOLYGON (((933100.918 192536.086, 933091...."
4,1,0.116357,0.000782,Newark Airport,1,EWR,1.0,02:00:00,0.123288,"MULTIPOLYGON (((933100.918 192536.086, 933091...."


In [55]:
# Avoiding disappering areas in the animation

# Locations
locations = gdf[["LocationID", "zone", "borough", "geometry"]].drop_duplicates()

# Times
times = pd.DataFrame({
    "time_stamp": sorted(gdf["time_stamp"].dropna().unique())
})

# Cartesian product
complete = locations.merge(times, how="cross")

# Merging the ride data
complete = complete.merge(
    gdf[["LocationID", "time_stamp", "avg_rides"]],
    on=["LocationID", "time_stamp"],
    how="left"
)

# Filling missing values
complete["avg_rides"] = complete["avg_rides"].fillna(0)

In [56]:
# Filtering a part of data for testing

filtered_gdf = complete[complete["time_stamp"] == '00:00:00']

geojson_filtered = filtered_gdf.geometry.__geo_interface__

In [57]:
# Creating a static map to test

fig = px.choropleth(
    filtered_gdf,
    geojson=geojson_filtered,
    locations=filtered_gdf.index,
    color="avg_rides",
    color_continuous_scale="Viridis",
    projection="mercator"
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [58]:
# Defining the GeoJSON representation of the GeoDataFrame

geojson = complete[["LocationID", "geometry"]].drop_duplicates("LocationID").__geo_interface__

In [59]:
# Creating the animated heatmap

# For performance measurement
import time

# Setting the default renderer to browser for better performance
import plotly.io as pio

pio.renderers.default = "browser"

# Creating the animated choropleth map

t1 = time.perf_counter()

# Creating figure
fig = px.choropleth(
    complete,
    geojson=geojson,
    locations="LocationID",
    featureidkey="properties.LocationID",
    color="avg_rides",
    range_color=(0, 300),
    animation_frame="time_stamp",
    color_continuous_scale="Turbo",
    projection="mercator",
    hover_name="zone",
    labels={
        "avg_rides": "Average rides",
        "borough": "Borough",
        "time_stamp": "Time"
    },
    hover_data={
        "LocationID": False,
        "zone": False,      # Don't repeat it since it's the hover title
        "borough": True,
        "time_stamp": True,
        "avg_rides": ":.1f"
    }
)

print("Figure:", time.perf_counter() - t1)

t2 = time.perf_counter()

# Setting the Layoout and Animation Settings
fig.update_geos(fitbounds="locations", visible=False)

fig.update_layout(
    title=dict(text="Number of Pickups — 00:00:00",
               x=0.5, 
               y=0.95, 
               xanchor="center", 
               yanchor="top",
               font={
                     "size": 24,
                     "family": "Arial",
                     "color": "black"
                    },),
    coloraxis_colorbar=dict(title="Avg Rides",
                            tickvals=[0, 50, 100, 150, 200, 250, 300],
                            ticktext=["0", "50", "100", "150", "200", "250", "300+"]),
    margin=dict(l=0, r=0, t=60, b=0)
)

for frame in fig.frames:
    frame.layout = {
        "title": f"Number of Pickups — {frame.name}"
    }

fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 200
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 0
fig.layout.sliders[0].currentvalue.prefix = "Time: "

print("Settings:", time.perf_counter() - t2)

t3 = time.perf_counter()

# Displaying the figure
fig.show()

print("Show:", time.perf_counter() - t3)
    

Figure: 9.356668699998409
Settings: 0.02095129992812872
Show: 17.68570140004158


In [60]:
# Saving the figure as an HTML file

fig.write_html(
    "../03_outputs/nyc_number_of_pickups.html",
    include_plotlyjs=True
)